<a href="https://colab.research.google.com/github/TimeiaVais/MLOps_Lab_1/blob/main/MLOps_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install libraries**

In [ ]:
!pip install torch torchvision scikit-learn pyyaml

# **Config**

In [ ]:
config = {
    "data": {
        "batch_size": 64
    },
    "training": {
        "epochs": 10,
        "learning_rate": 0.001
    },
    "model": {
        "num_classes": 10
    }
}

# **Logging**

In [ ]:
import logging

logging.basicConfig(
    filename="training.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logging.info("Pipeline started")

# **Download + Data Augmentation + Split**

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

def download_and_prepare_data(batch_size):
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor()
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor()
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size

    train_dataset, val_dataset = random_split(
        train_dataset,
        [train_size, val_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader

# **CNN Model**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(-1, 64 * 8 * 8)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# **Evaluation function**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.numpy())

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    return total_loss, acc, precision, recall, f1

# **Training Pipeline**

In [ ]:
def train_pipeline(config):
    logging.info("Training started")

    train_loader, val_loader, test_loader = download_and_prepare_data(
        config["data"]["batch_size"]
    )

    model = SimpleCNN(config["model"]["num_classes"])

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["training"]["learning_rate"]
    )

    best_val_loss = float("inf")

    for epoch in range(config["training"]["epochs"]):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        val_loss, acc, prec, rec, f1 = evaluate_model(
            model, val_loader, criterion
        )

        logging.info(f"Epoch {epoch}")
        logging.info(f"Train Loss {train_loss}")
        logging.info(f"Val Loss {val_loss}")
        logging.info(f"Accuracy {acc}")

        print(f"Epoch {epoch}")
        print(f"Train Loss {train_loss}")
        print(f"Val Loss {val_loss}")
        print(f"Accuracy {acc}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pth")

    print("Testing best model...")
    model.load_state_dict(torch.load("best_model.pth"))

    test_loss, acc, prec, rec, f1 = evaluate_model(
        model, test_loader, criterion
    )

    print("TEST RESULTS")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)

    logging.info("Training finished")

# **Run pipeline**

In [ ]:
train_pipeline(config)

100%|██████████| 170M/170M [00:02<00:00, 73.7MB/s]


Epoch 0
Train Loss 957.8399861454964
Val Loss 209.92989885807037
Accuracy 0.5284
Epoch 1
Train Loss 739.4664797186852
Val Loss 179.11299061775208
Accuracy 0.6007
Epoch 2
Train Loss 646.7076750397682
Val Loss 158.04384928941727
Accuracy 0.6482
Epoch 3
Train Loss 585.415359556675
Val Loss 158.0311518907547
Accuracy 0.6508
Epoch 4
Train Loss 537.9848706126213
Val Loss 144.25666844844818
Accuracy 0.6777
Epoch 5
Train Loss 504.25508081912994
Val Loss 140.95708572864532
Accuracy 0.6901
Epoch 6
Train Loss 474.09681582450867
Val Loss 133.36512351036072
Accuracy 0.7029
Epoch 7
Train Loss 443.35823148489
Val Loss 128.4260661303997
Accuracy 0.7178
Epoch 8
Train Loss 420.48964136838913
Val Loss 130.0218697488308
Accuracy 0.7194
Epoch 9
Train Loss 401.0137719810009
Val Loss 130.17075231671333
Accuracy 0.7196
Testing best model...
TEST RESULTS
Accuracy: 0.7288
Precision: 0.7265553859786943
Recall: 0.7288
F1: 0.7259794323655954
